Random Forest

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # keep numeric columns only
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # drop missing values
    df_phase = df_phase.dropna()

    # features and labels
    X = df_phase.drop("label", axis=1).values
    y = df_phase["label"].values

    print("Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # train test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    # feature scaling
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # random forest model
    model = RandomForestClassifier(
        n_estimators=400,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # predictions
    THRESHOLD = 0.40

    y_prob = model.predict_proba(X_test)[:, 1]

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # roc auc
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # confusion matrix
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return model, roc

# filter phase-wise data
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Shape: (141, 132)
Class Distribution: [99 42]

Predicted Distribution:
[22  7]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.75      0.71        20
           1       0.29      0.22      0.25         9

    accuracy                           0.59        29
   macro avg       0.48      0.49      0.48        29
weighted avg       0.56      0.59      0.57        29

ROC-AUC Score: 0.6028

Confusion Matrix:
[[15  5]
 [ 7  2]]

------------------------------------------------------------

Random Forest 5 folds

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# training function (5-fold rf)
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # keep numeric columns only
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # drop missing values
    df_phase = df_phase.dropna()

    # features and labels
    X = df_phase.drop("label", axis=1).values
    y = df_phase["label"].values

    print("Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # random forest model
        model = RandomForestClassifier(
            n_estimators=400,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        THRESHOLD = 0.40

        y_prob = model.predict_proba(X_test)[:, 1]

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter phase-wise data
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Shape: (141, 132)
Class Distribution: [99 42]

----- Fold 1 -----
Fold ROC-AUC: 0.6278

----- Fold 2 -----
Fold ROC-AUC: 0.8094

----- Fold 3 -----
Fold ROC-AUC: 0.6531

----- Fold 4 -----
Fold ROC-AUC: 0.7188

----- Fold 5 -----
Fold ROC-AUC: 0.6579

------------------------------------------------------------
FINAL RESULTS FOR PHASE 1
------------------------------------------------------------

Fold ROC-AUC Scores:
[0.6278 0.8094 0.6531 0.7188 0.6579]

Mean Fold ROC-AUC:
0.6934

Overall ROC-AUC:
0.6762

Classification Report:
              precision    recall  f1-score   

Random Forest 10 folds

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# training function (5-fold rf)
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # keep numeric columns only
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # drop missing values
    df_phase = df_phase.dropna()

    # features and labels
    X = df_phase.drop("label", axis=1).values
    y = df_phase["label"].values

    print("Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # 10-fold stratified cv
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # random forest model
        model = RandomForestClassifier(
            n_estimators=400,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        THRESHOLD = 0.40

        y_prob = model.predict_proba(X_test)[:, 1]

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter phase-wise data
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Shape: (141, 132)
Class Distribution: [99 42]

----- Fold 1 -----
Fold ROC-AUC: 0.64

----- Fold 2 -----
Fold ROC-AUC: 0.5

----- Fold 3 -----
Fold ROC-AUC: 0.6625

----- Fold 4 -----
Fold ROC-AUC: 0.8

----- Fold 5 -----
Fold ROC-AUC: 0.7875

----- Fold 6 -----
Fold ROC-AUC: 0.65

----- Fold 7 -----
Fold ROC-AUC: 0.6875

----- Fold 8 -----
Fold ROC-AUC: 0.7875

----- Fold 9 -----
Fold ROC-AUC: 0.7

----- Fold 10 -----
Fold ROC-AUC: 0.5556

------------------------------------------------------------
FINAL RESULTS FOR PHASE 1
-------------------------------------------------

XGBoost

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    c for c in df.columns
    if c not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # keep only numeric
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # features & labels
    X = df_phase.drop("label", axis=1)
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # handle missing values
    imputer = SimpleImputer(strategy="median")

    X = imputer.fit_transform(X)

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.0001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # select best features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(80, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # train test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # handle class imbalance
    neg = np.sum(y_train == 0)
    pos = np.sum(y_train == 1)

    scale_pos_weight = neg / pos

    print("Scale Pos Weight:", round(scale_pos_weight, 2))

    # tuned xgboost model
    model = XGBClassifier(

        # core
        objective="binary:logistic",
        eval_metric="auc",

        # trees
        n_estimators=400,
        max_depth=4,

        # learning
        learning_rate=0.03,

        # sampling
        subsample=0.8,
        colsample_bytree=0.7,

        # regularization
        gamma=1,
        min_child_weight=3,

        reg_alpha=1,
        reg_lambda=3,

        # imbalance handling
        scale_pos_weight=scale_pos_weight,

        # faster optimized training
        tree_method="hist",

        random_state=42,
        n_jobs=-1
    )

    # train model
    model.fit(X_train, y_train)

    # predict probabilities
    y_prob = model.predict_proba(X_test)[:, 1]

    # threshold
    THRESHOLD = 0.30

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # roc-auc
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # confusion matrix
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return model, roc

# filter phase-wise data
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 132)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 114)
Shape After SelectKBest: (141, 80)
Scale Pos Weight: 2.39

Predicted Distribution:
[12 17]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.45      0.56        20
           1       0.35      0.67      0.46         9

    accuracy                           0.52        29
   macro avg       0.55      0.56      0.51        29
weighted avg       0.63      0.52      0.53        29

ROC-AUC Sc

XGBoost 5 Folds

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # keep numeric only
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase.drop("label", axis=1)
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    selector = VarianceThreshold(
        threshold=0.001
    )

    X = selector.fit_transform(X)

    print("Shape After Variance Threshold:", X.shape)

    # handle class imbalance
    neg = np.sum(y == 0)
    pos = np.sum(y == 1)

    scale_pos_weight = neg / pos

    print("Scale Pos Weight:", round(scale_pos_weight, 2))

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # xgboost model
        model = XGBClassifier(

            objective="binary:logistic",
            eval_metric="auc",

            # trees
            n_estimators=300,
            max_depth=4,

            # learning
            learning_rate=0.03,

            # sampling
            subsample=0.8,
            colsample_bytree=0.7,

            # regularization
            gamma=1,
            min_child_weight=3,

            # imbalance handling
            scale_pos_weight=scale_pos_weight,

            # misc
            random_state=42,
            n_jobs=-1
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.40

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter phase-wise data
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 132)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 107)
Scale Pos Weight: 2.36

----- Fold 1 -----
Fold ROC-AUC: 0.6222

----- Fold 2 -----
Fold ROC-AUC: 0.7125

----- Fold 3 -----
Fold ROC-AUC: 0.6562

----- Fold 4 -----
Fold ROC-AUC: 0.75

----- Fold 5 -----
Fold ROC-AUC: 0.6725

------------------------------------------------------------
FINAL RESULTS FOR PHASE 1
------------------------------------------------------------

Fold ROC-AUC Scores:
[0.6222 0.7125 0.6562 0.75   0.6725]

Mean Fold ROC-AUC:
0.6827

Overall ROC-AUC:
0.66

XGBoost 10 Folds

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from xgboost import XGBClassifier

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # keep numeric only
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase.drop("label", axis=1)
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    selector = VarianceThreshold(
        threshold=0.001
    )

    X = selector.fit_transform(X)

    print("Shape After Variance Threshold:", X.shape)

    # handle class imbalance
    neg = np.sum(y == 0)
    pos = np.sum(y == 1)

    scale_pos_weight = neg / pos

    print("Scale Pos Weight:", round(scale_pos_weight, 2))

    # 10-fold stratified cv
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # xgboost model
        model = XGBClassifier(

            objective="binary:logistic",
            eval_metric="auc",

            # trees
            n_estimators=300,
            max_depth=4,

            # learning
            learning_rate=0.03,

            # sampling
            subsample=0.8,
            colsample_bytree=0.7,

            # regularization
            gamma=1,
            min_child_weight=3,

            # imbalance handling
            scale_pos_weight=scale_pos_weight,

            # misc
            random_state=42,
            n_jobs=-1
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.40

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter phase-wise data
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 132)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 107)
Scale Pos Weight: 2.36

----- Fold 1 -----
Fold ROC-AUC: 0.74

----- Fold 2 -----
Fold ROC-AUC: 0.45

----- Fold 3 -----
Fold ROC-AUC: 0.7

----- Fold 4 -----
Fold ROC-AUC: 0.7

----- Fold 5 -----
Fold ROC-AUC: 0.8

----- Fold 6 -----
Fold ROC-AUC: 0.75

----- Fold 7 -----
Fold ROC-AUC: 0.825

----- Fold 8 -----
Fold ROC-AUC: 0.7

----- Fold 9 -----
Fold ROC-AUC: 0.6

----- Fold 10 -----
Fold ROC-AUC: 0.6444

------------------------------------------------------------
FINAL RES

SVM

In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # keep numeric columns only
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # drop missing values
    df_phase = df_phase.dropna()

    # features and labels
    X = df_phase.drop("label", axis=1).values
    y = df_phase["label"].values

    print("Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # train test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # feature scaling
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # svm model
    model = SVC(

        # rbf kernel
        kernel="rbf",

        # needed for roc-auc
        probability=True,

        # handle imbalance
        class_weight="balanced",

        # tuned parameters
        C=2,

        gamma="scale",

        random_state=42
    )

    # train model
    model.fit(X_train, y_train)

    # predict
    THRESHOLD = 0.40

    y_prob = model.predict_proba(X_test)[:, 1]

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # roc-auc
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # confusion matrix
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return model, roc

# filter phase-wise data
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Shape: (141, 132)
Class Distribution: [99 42]

Predicted Distribution:
[25  4]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.85      0.76        20
           1       0.25      0.11      0.15         9

    accuracy                           0.62        29
   macro avg       0.47      0.48      0.45        29
weighted avg       0.55      0.62      0.57        29

ROC-AUC Score: 0.5444

Confusion Matrix:
[[17  3]
 [ 8  1]]

------------------------------------------------------------

SVM 5 folds

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # keep numeric only
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase.drop("label", axis=1)
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    selector = VarianceThreshold(
        threshold=0.002
    )

    X = selector.fit_transform(X)

    print("Shape After Variance Threshold:", X.shape)

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # tuned svm model
        model = SVC(

            # nonlinear separation
            kernel="rbf",

            # needed for roc-auc
            probability=True,

            # handle imbalance
            class_weight="balanced",

            # stronger fitting
            C=3,

            # smoother boundary
            gamma=0.0005,

            random_state=42
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        THRESHOLD = 0.40

        y_prob = model.predict_proba(X_test)[:, 1]

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter phase-wise data
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 132)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 105)

----- Fold 1 -----
Fold ROC-AUC: 0.5722

----- Fold 2 -----
Fold ROC-AUC: 0.675

----- Fold 3 -----
Fold ROC-AUC: 0.6813

----- Fold 4 -----
Fold ROC-AUC: 0.7437

----- Fold 5 -----
Fold ROC-AUC: 0.6959

------------------------------------------------------------
FINAL RESULTS FOR PHASE 1
------------------------------------------------------------

Fold ROC-AUC Scores:
[0.5722 0.675  0.6812 0.7437 0.6959]

Mean Fold ROC-AUC:
0.6736

Overall ROC-AUC:
0.67

Classification Repor

SVM 10 folds

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.svm import SVC

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [col for col in df.columns if col not in remove_cols]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # keep numeric only
    df_phase = df_phase.select_dtypes(
        include=["int64", "float64"]
    )

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase.drop("label", axis=1)
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    selector = VarianceThreshold(
        threshold=0.002
    )

    X = selector.fit_transform(X)

    print("Shape After Variance Threshold:", X.shape)

    # 10-fold stratified cv
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # tuned svm model
        model = SVC(

            # nonlinear separation
            kernel="rbf",

            # needed for roc-auc
            probability=True,

            # handle imbalance
            class_weight="balanced",

            # stronger fitting
            C=3,

            # smoother boundary
            gamma=0.0005,

            random_state=42
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        THRESHOLD = 0.40

        y_prob = model.predict_proba(X_test)[:, 1]

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter phase-wise data
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 132)
Class Distribution: [99 42]
Shape After Variance Threshold: (141, 105)

----- Fold 1 -----
Fold ROC-AUC: 0.62

----- Fold 2 -----
Fold ROC-AUC: 0.5

----- Fold 3 -----
Fold ROC-AUC: 0.65

----- Fold 4 -----
Fold ROC-AUC: 0.7

----- Fold 5 -----
Fold ROC-AUC: 0.15

----- Fold 6 -----
Fold ROC-AUC: 0.675

----- Fold 7 -----
Fold ROC-AUC: 0.725

----- Fold 8 -----
Fold ROC-AUC: 0.725

----- Fold 9 -----
Fold ROC-AUC: 0.525

----- Fold 10 -----
Fold ROC-AUC: 0.5778

------------------------------------------------------------
FINAL RESULTS FOR PHASE 1


Logistic Regression

In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.0005
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # select best features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(120, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # train test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # feature scaling
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # tuned logistic regression
    model = LogisticRegression(

        # stronger regularization tuning
        C=0.3,

        # handles imbalance
        class_weight="balanced",

        # better convergence
        solver="liblinear",

        # more iterations
        max_iter=1000,

        random_state=42
    )

    # train model
    model.fit(X_train, y_train)

    # predict probabilities
    y_prob = model.predict_proba(X_test)[:, 1]

    # threshold
    THRESHOLD = 0.35

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # roc-auc
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # confusion matrix
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return model, roc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 131)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 108)
Shape After SelectKBest: (141, 108)

Predicted Distribution:
[13 16]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.45      0.55        20
           1       0.31      0.56      0.40         9

    accuracy                           0.48        29
   macro avg       0.50      0.50      0.47        29
weighted avg       0.57      0.48      0.50        29

ROC-AUC Score: 0.5333

Confusion

Logistic Regression 5 folds

In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.0005
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # select best features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(120, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # logistic regression model
        model = LogisticRegression(

            # better generalization
            C=0.3,

            # handle imbalance
            class_weight="balanced",

            # stable solver
            solver="liblinear",

            # better convergence
            max_iter=1000,

            random_state=42
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.35

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 131)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 108)
Shape After SelectKBest: (141, 108)

----- Fold 1 -----
Fold ROC-AUC: 0.6556

----- Fold 2 -----
Fold ROC-AUC: 0.4938

----- Fold 3 -----
Fold ROC-AUC: 0.7812

----- Fold 4 -----
Fold ROC-AUC: 0.6938

----- Fold 5 -----
Fold ROC-AUC: 0.6082

------------------------------------------------------------
FINAL RESULTS FOR PHASE 1
------------------------------------------------------------

Fold ROC-AUC Scores:
[0.6556 0.4938 0.7812 0.6938 0.6082]

Mean Fold ROC-AUC:
0.6465

Overa

Logistic Regression 10 Folds

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.0005
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # select best features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(120, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 10-fold stratified cv
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # feature scaling
        scaler = StandardScaler()

        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        # logistic regression model
        model = LogisticRegression(

            # better generalization
            C=0.3,

            # handle imbalance
            class_weight="balanced",

            # stable solver
            solver="liblinear",

            # better convergence
            max_iter=1000,

            random_state=42
        )

        # train model
        model.fit(X_train, y_train)

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.35

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 131)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 108)
Shape After SelectKBest: (141, 108)

----- Fold 1 -----
Fold ROC-AUC: 0.58

----- Fold 2 -----
Fold ROC-AUC: 0.6

----- Fold 3 -----
Fold ROC-AUC: 0.425

----- Fold 4 -----
Fold ROC-AUC: 0.65

----- Fold 5 -----
Fold ROC-AUC: 0.775

----- Fold 6 -----
Fold ROC-AUC: 0.9

----- Fold 7 -----
Fold ROC-AUC: 0.55

----- Fold 8 -----
Fold ROC-AUC: 0.625

----- Fold 9 -----
Fold ROC-AUC: 0.35

----- Fold 10 -----
Fold ROC-AUC: 0.5111

---------------------------------------------------

CatBoost

In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from catboost import CatBoostClassifier

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important audio features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(80, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # train test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )

    # handle class imbalance
    neg = np.sum(y_train == 0)
    pos = np.sum(y_train == 1)

    class_weights = [
        1,
        neg / pos
    ]

    print("Class Weights:", class_weights)

    # balanced audio catboost
    model = CatBoostClassifier(

        # core
        loss_function="Logloss",
        eval_metric="AUC",

        # moderate boosting
        iterations=500,

        # moderate depth
        depth=5,

        # balanced learning
        learning_rate=0.025,

        # slightly stronger regularization
        l2_leaf_reg=7,

        # lower randomness
        random_strength=1,

        # better generalization
        bootstrap_type="Bernoulli",
        subsample=0.8,

        # handle imbalance
        class_weights=class_weights,

        # stable trees
        grow_policy="SymmetricTree",

        random_seed=42,

        verbose=0
    )

    # train model
    model.fit(
        X_train,
        y_train
    )

    # predict probabilities
    y_prob = model.predict_proba(X_test)[:, 1]

    # threshold
    THRESHOLD = 0.35

    y_pred = (y_prob > THRESHOLD).astype(int)

    print("\nPredicted Distribution:")
    print(np.bincount(y_pred))

    print("\nActual Distribution:")
    print(np.bincount(y_test))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        zero_division=0
    ))

    # roc-auc
    roc = roc_auc_score(y_test, y_prob)

    print("ROC-AUC Score:", round(roc, 4))

    # confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    return model, roc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
model_p1, roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

model_p2, roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

model_p3, roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 131)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 127)
Shape After SelectKBest: (141, 80)
Class Weights: [1, np.float64(2.393939393939394)]

Predicted Distribution:
[15 14]

Actual Distribution:
[20  9]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.55      0.63        20
           1       0.36      0.56      0.43         9

    accuracy                           0.55        29
   macro avg       0.55      0.55      0.53        29
weighted avg       0.62      0.55     

CatBoost 5 folds

In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from catboost import CatBoostClassifier

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important audio features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(80, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 5-fold stratified cv
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # handle class imbalance
        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)

        class_weights = [
            1,
            neg / pos
        ]

        # balanced audio catboost
        model = CatBoostClassifier(

            # core
            loss_function="Logloss",
            eval_metric="AUC",

            # moderate boosting
            iterations=500,

            # moderate depth
            depth=5,

            # balanced learning
            learning_rate=0.025,

            # slightly stronger regularization
            l2_leaf_reg=7,

            # lower randomness
            random_strength=1,

            # better generalization
            bootstrap_type="Bernoulli",
            subsample=0.8,

            # handle imbalance
            class_weights=class_weights,

            # stable trees
            grow_policy="SymmetricTree",

            random_seed=42,

            verbose=0
        )

        # train model
        model.fit(
            X_train,
            y_train
        )

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.35

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 131)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 127)
Shape After SelectKBest: (141, 80)

----- Fold 1 -----
Fold ROC-AUC: 0.5389

----- Fold 2 -----
Fold ROC-AUC: 0.775

----- Fold 3 -----
Fold ROC-AUC: 0.6687

----- Fold 4 -----
Fold ROC-AUC: 0.6937

----- Fold 5 -----
Fold ROC-AUC: 0.5906

------------------------------------------------------------
FINAL RESULTS FOR PHASE 1
------------------------------------------------------------

Fold ROC-AUC Scores:
[0.5389 0.775  0.6688 0.6938 0.5906]

Mean Fold ROC-AUC:
0.6534

Overall

CatBoost 10 folds

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif
)

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

from catboost import CatBoostClassifier

# load dataset
df = pd.read_csv("../../data/audio_features_phasewise.csv")

print("-" * 60)
print("DATASET INFORMATION")
print("-" * 60)

print("Dataset Shape:", df.shape)

# normalize phase column
df["phase"] = df["phase"].astype(str).str.lower().str.strip()

print("\nAvailable Phases:")
print(df["phase"].unique())

# remove non-feature columns
remove_cols = [
    "participant_id",
    "phase",
    "label"
]

feature_cols = [
    col for col in df.columns
    if col not in remove_cols
]

print("\nTotal Features:", len(feature_cols))

# training function
def train_phase_model(df_phase, phase_name):

    print("\n" + "-" * 60)
    print(f"TRAINING FOR {phase_name}")
    print("-" * 60)

    # drop missing values
    df_phase = df_phase.dropna()

    # features & labels
    X = df_phase[feature_cols]
    y = df_phase["label"].values

    print("Original Shape:", X.shape)
    print("Class Distribution:", np.bincount(y))

    # remove low variance features
    variance_selector = VarianceThreshold(
        threshold=0.00001
    )

    X = variance_selector.fit_transform(X)

    print("\nShape After Variance Threshold:", X.shape)

    # keep important audio features
    kbest_selector = SelectKBest(
        score_func=mutual_info_classif,
        k=min(80, X.shape[1])
    )

    X = kbest_selector.fit_transform(X, y)

    print("Shape After SelectKBest:", X.shape)

    # 10-fold stratified cv
    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    # store predictions
    all_probs = np.zeros(len(y))
    all_preds = np.zeros(len(y))

    fold_aucs = []

    # train each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

        print(f"\n----- Fold {fold} -----")

        X_train, X_test = X[train_idx], X[test_idx]

        y_train, y_test = y[train_idx], y[test_idx]

        # handle class imbalance
        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)

        class_weights = [
            1,
            neg / pos
        ]

        # balanced audio catboost
        model = CatBoostClassifier(

            # core
            loss_function="Logloss",
            eval_metric="AUC",

            # moderate boosting
            iterations=500,

            # moderate depth
            depth=5,

            # balanced learning
            learning_rate=0.025,

            # slightly stronger regularization
            l2_leaf_reg=7,

            # lower randomness
            random_strength=1,

            # better generalization
            bootstrap_type="Bernoulli",
            subsample=0.8,

            # handle imbalance
            class_weights=class_weights,

            # stable trees
            grow_policy="SymmetricTree",

            random_seed=42,

            verbose=0
        )

        # train model
        model.fit(
            X_train,
            y_train
        )

        # predict
        y_prob = model.predict_proba(X_test)[:, 1]

        THRESHOLD = 0.35

        y_pred = (y_prob > THRESHOLD).astype(int)

        # store predictions
        all_probs[test_idx] = y_prob
        all_preds[test_idx] = y_pred

        # fold roc-auc
        fold_auc = roc_auc_score(y_test, y_prob)

        fold_aucs.append(fold_auc)

        print("Fold ROC-AUC:", round(fold_auc, 4))

    # final results
    overall_auc = roc_auc_score(y, all_probs)

    print("\n" + "-" * 60)
    print(f"FINAL RESULTS FOR {phase_name}")
    print("-" * 60)

    print("\nFold ROC-AUC Scores:")
    print(np.round(fold_aucs, 4))

    print("\nMean Fold ROC-AUC:")
    print(round(np.mean(fold_aucs), 4))

    print("\nOverall ROC-AUC:")
    print(round(overall_auc, 4))

    # classification report
    print("\nClassification Report:")
    print(classification_report(
        y,
        all_preds,
        zero_division=0
    ))

    # confusion matrix
    cm = confusion_matrix(y, all_preds)

    print("\nConfusion Matrix:")
    print(cm)

    return overall_auc

# filter data phase-wise
df_p1 = df[df["phase"] == "phase1"]

df_p2 = df[df["phase"] == "phase2"]

df_p3 = df[df["phase"] == "phase3"]

# train models
roc_p1 = train_phase_model(
    df_p1,
    "PHASE 1"
)

roc_p2 = train_phase_model(
    df_p2,
    "PHASE 2"
)

roc_p3 = train_phase_model(
    df_p3,
    "PHASE 3"
)

# final summary
print("\n" + "-" * 60)
print("FINAL ROC-AUC SCORES")
print("-" * 60)

print(f"Phase 1 ROC-AUC: {roc_p1:.4f}")
print(f"Phase 2 ROC-AUC: {roc_p2:.4f}")
print(f"Phase 3 ROC-AUC: {roc_p3:.4f}")

------------------------------------------------------------
DATASET INFORMATION
------------------------------------------------------------
Dataset Shape: (423, 134)

Available Phases:
<StringArray>
['phase1', 'phase2', 'phase3']
Length: 3, dtype: str

Total Features: 131

------------------------------------------------------------
TRAINING FOR PHASE 1
------------------------------------------------------------
Original Shape: (141, 131)
Class Distribution: [99 42]

Shape After Variance Threshold: (141, 127)
Shape After SelectKBest: (141, 80)

----- Fold 1 -----
Fold ROC-AUC: 0.64

----- Fold 2 -----
Fold ROC-AUC: 0.475

----- Fold 3 -----
Fold ROC-AUC: 0.775

----- Fold 4 -----
Fold ROC-AUC: 0.625

----- Fold 5 -----
Fold ROC-AUC: 0.825

----- Fold 6 -----
Fold ROC-AUC: 0.55

----- Fold 7 -----
Fold ROC-AUC: 0.625

----- Fold 8 -----
Fold ROC-AUC: 0.7

----- Fold 9 -----
Fold ROC-AUC: 0.575

----- Fold 10 -----
Fold ROC-AUC: 0.4444

------------------------------------------------